<a href="https://colab.research.google.com/github/nickpartner/mammogram/blob/main/mammogram_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Information

This notebook describes the code to execute the training and setup for the kaggle Abdulaziz mammogram dataset.

Whilst it should run end-to-end please note that some cells show at times multiple versions due to constraints:

a. Optimisation of code - partly to show working
b. Attempts to work around CoLab resource constraints

At times this means that ocassional issues occur when switching between different processor models [CPU/TPU], so please be kind.

The images are around 3000 and are processed multiple times. Each layer has been stored back to avoid multiple processing of these - but a clean run may take time due to the size!

# Setup Environment

In [ ]:
# mount gdrive so we can download there and persist

from google.colab import drive
import os

# Mount Google Drive
drive.mount('/gdrive', force_remount=True)

base_path = '/gdrive/MyDrive/assignment/abdulaziz'
#base_path = '/home/abdulaziz'

Mounted at /gdrive


## Grab the Data

In [ ]:
# This data set comes from the King Abdulaziz University dataset

# The current study aims to build the first digitalized mammogram dataset for breast cancer in Saudi Arabia,
# depend on the BI-RADS categories, to solve the availability problem of local public datasets by collecting,
# categorizing, and annotating mammogram images, supporting the medical field by providing physicians with different
# diagnosed cases especially in Saudi Arabia

# https://www.kaggle.com/datasets/asmaasaad/king-abdulaziz-university-mammogram-dataset/data

The dataset is setup based on the BI-RADS classification system. This is is a 7 point scale when reading a mamogram.

https://www.wellwomen.co.za/helpie_faq/what-is-the-meaning-of-the-word-bi-rads-which-i-have-read-on-my-report/

The dataset contains

Character of background tissue:
 1 - Negative
 3 - Probably Benign [ 2 is benign ]
 4 - Suspicious
 5 - Highly Suggestive

Absent is
 0 - Scan is problematic and needs redoing
 2 - Benign
 6 - confirmed through biopsy



In [ ]:
# Grab the data from kagglehub into

import kagglehub
import os

base_dir = os.path.join(base_path,'base')

# kaggle likes to cache to a download directly which is annoying and ignores
# output directory at times
download_path = kagglehub.dataset_download("asmaasaad/king-abdulaziz-university-mammogram-dataset", output_dir=base_dir)

print("Path to dataset files:", download_path)
print("Base dir:", base_dir)


Using Colab cache for faster access to the 'king-abdulaziz-university-mammogram-dataset' dataset.
Path to dataset files: /kaggle/input/king-abdulaziz-university-mammogram-dataset
Base dir: /gdrive/MyDrive/assignment/abdulaziz/base


## Generate the metadata

In [ ]:
import requests
import pandas as pd

In [ ]:
# use the directory listings to construct the labelling information

import os

b1 = os.listdir(base_dir+'/BIRAD1/b1')
b3 = os.listdir(base_dir+'/Birad3/b3')
b4 = os.listdir(base_dir+'/Birad4/b4')
b5 = os.listdir(base_dir+'/Birad5/Birad5')


b1_df = pd.DataFrame(b1, columns=['filename'])
b1_df['BIRAD'] = 1
b1_df['path'] = 'BIRAD1/b1/'+b1_df['filename']

b3_df = pd.DataFrame(b3, columns=['filename'])
b3_df['BIRAD'] = 3
b3_df['path'] = 'Birad3/b3/'+b3_df['filename']

b4_df = pd.DataFrame(b4, columns=['filename'])
b4_df['BIRAD'] = 4
b4_df['path'] = 'Birad4/b4/'+b4_df['filename']

b5_df = pd.DataFrame(b5, columns=['filename'])
b5_df['BIRAD'] = 5
b5_df['path'] = 'Birad5/Birad5/'+b5_df['filename']


all_images = pd.concat([b1_df, b3_df, b4_df, b5_df])

all_images.head()


,filename,BIRAD,path
0,2017_BC009921_ MLO_R.jpg,1,BIRAD1/b1/2017_BC009921_ MLO_R.jpg
1,2017_BC010161_ CC_L.jpg,1,BIRAD1/b1/2017_BC010161_ CC_L.jpg
2,2017_BC010161_ CC_R.jpg,1,BIRAD1/b1/2017_BC010161_ CC_R.jpg
3,2017_BC010161_ MLO_L.jpg,1,BIRAD1/b1/2017_BC010161_ MLO_L.jpg
4,2017_BC010161_ MLO_R.jpg,1,BIRAD1/b1/2017_BC010161_ MLO_R.jpg


In [ ]:
# as a result we will use the file names which contain the patient id and image type
# to construct the labelled data

# date_patientno_imagetype_side

metadata = all_images['filename'].str.split('_',expand=True)

#metadata.head()

# side has the image file extension after - so will split that out.
# will keep the image type at the end, just so we can check we have consistent images

metadata_end = metadata[3].str.split('.',expand=True)

labels = all_images
labels['year'] = metadata[0]
labels['patient'] = metadata[1]
labels['imaging_type'] = metadata[2]
labels['side'] = metadata_end[0]
labels['file_type'] = metadata_end[1]

labels.head()

,filename,BIRAD,path,year,patient,imaging_type,side,file_type
0,2017_BC009921_ MLO_R.jpg,1,BIRAD1/b1/2017_BC009921_ MLO_R.jpg,2017,BC009921,MLO,R,jpg
1,2017_BC010161_ CC_L.jpg,1,BIRAD1/b1/2017_BC010161_ CC_L.jpg,2017,BC010161,CC,L,jpg
2,2017_BC010161_ CC_R.jpg,1,BIRAD1/b1/2017_BC010161_ CC_R.jpg,2017,BC010161,CC,R,jpg
3,2017_BC010161_ MLO_L.jpg,1,BIRAD1/b1/2017_BC010161_ MLO_L.jpg,2017,BC010161,MLO,L,jpg
4,2017_BC010161_ MLO_R.jpg,1,BIRAD1/b1/2017_BC010161_ MLO_R.jpg,2017,BC010161,MLO,R,jpg


## Test Paths

Even though the sample sizes aren't huge we are going to setup some sample paths to help check the code and provide examples against without iterating through 000's of image manipulations each time.

Yes, these might be a little bit path specific, but are really debug mainly

In [ ]:
test_paths_b1 = ['BIRAD1/b1/2017_BC009804_ MLO_R.jpg']
test_paths_b5 = ['Birad5/Birad5/2018_BC0022025_ MLO_R.jpg']


# Data Labels

check for oddities and issues and remove anything prior to features and training

### duplicate images

In [ ]:
## Noticed later that there are duplicate images with different BI-RADS
## they are the same patient and also manual review shows the same image
## but clipped. There are only two patients so manually removing these.
## Without a specialist to recategorise, this isn't possible.
##
## (Note: girlfriend refused to help with this despite being a doctor)

file_count = labels \
  .groupby(['filename'])['filename']\
  .count().reset_index(name='count')

duplicates = file_count \
                .loc[file_count['count']>1]


## remove duplicates - left_anti is being mean to me, so a little convoluted!

no_duplicates_join = pd.merge(labels, duplicates, on='filename', how='left') \

no_duplicates = no_duplicates_join \
                    .loc[no_duplicates_join['count'].isna()] \
                    .drop(['count'],axis=1)

display(no_duplicates)

,filename,BIRAD,path,year,patient,imaging_type,side,file_type
0,2017_BC009921_ MLO_R.jpg,1,BIRAD1/b1/2017_BC009921_ MLO_R.jpg,2017,BC009921,MLO,R,jpg
1,2017_BC010161_ CC_L.jpg,1,BIRAD1/b1/2017_BC010161_ CC_L.jpg,2017,BC010161,CC,L,jpg
2,2017_BC010161_ CC_R.jpg,1,BIRAD1/b1/2017_BC010161_ CC_R.jpg,2017,BC010161,CC,R,jpg
3,2017_BC010161_ MLO_L.jpg,1,BIRAD1/b1/2017_BC010161_ MLO_L.jpg,2017,BC010161,MLO,L,jpg
4,2017_BC010161_ MLO_R.jpg,1,BIRAD1/b1/2017_BC010161_ MLO_R.jpg,2017,BC010161,MLO,R,jpg
...,...,...,...,...,...,...,...,...
2371,2018_BC0022603_ MLO_R.jpg,5,Birad5/Birad5/2018_BC0022603_ MLO_R.jpg,2018,BC0022603,MLO,R,jpg
2372,2018_BC005421_ CC_L.jpg,5,Birad5/Birad5/2018_BC005421_ CC_L.jpg,2018,BC005421,CC,L,jpg
2374,2018_BC005421_ MLO_L.jpg,5,Birad5/Birad5/2018_BC005421_ MLO_L.jpg,2018,BC005421,MLO,L,jpg
2376,2019_BC0024763_ CC_L.jpg,5,Birad5/Birad5/2019_BC0024763_ CC_L.jpg,2019,BC0024763,CC,L,jpg


### Remove patient data

Spotting that some of the images include patient data. To avoid this processing I'm manually removing those patient numbers

In [ ]:
pii = {"bad_patient": ['BC010801','BC003105','BC017061','BC002922','BC008703','BC23801']}

pii_df = pd.DataFrame(data=pii)

## remove duplicates - left_anti is being mean to me, so a little convoluted!

no_pii_join = pd.merge(no_duplicates, pii_df, left_on='patient', right_on='bad_patient', how='left') \

no_pii = no_pii_join \
                    .loc[no_pii_join['bad_patient'].isnull()] \
                    .drop(['bad_patient'],axis=1)


In [ ]:
## used as a final dataframe to allow for multiple cleaning steps inserted as we go!

final_labels = no_pii.copy()

# Image Cleaning

#### image xif metadata

In [ ]:
## lets avoid running this 00000 of times!

!pip install Pillow



ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
# lets grab the details of the image as well as this might be useful for a bit
# of analysis - trimming, clipping etc.

# we can grab out the image type, size and format to verify

from PIL import Image

image_details = pd.DataFrame(columns=('path','type'))

for image_path in final_labels['path']:

  source_full_path = os.path.join(download_path,image_path)

  im = Image.open(source_full_path)

  im_details = pd.DataFrame({
                                'file':[im.filename],
                                'type':[im.format],
                                'width':[im.width],
                                'height':[im.height],
                                'filename':image_path.split("/")[-1]
                            })
  image_details = pd.concat([image_details, im_details])
  im.close()

image_details.head()

#### image orientation

Left and right breast images have a different orientation due to the way that the images are made. The images with 'L' have the chest wall against the left of the image and empty space to the right and 'R' has the inverse. This may cause some problems with CNN which works on slices and the slices will appear very differently on ordering so aligning orientation may boost.

Note that this will apply to both CC and MLO.

It might be that later we need to duplicate or retranspose these to increase sample sizes, but for now it provides some consistency (particularly for the next part)

In [ ]:
from PIL import Image
import os

# will flip all of the rights to left.

# Define the output directory for processed images
# we have a base_path and then relative folder for feature

output_dir = os.path.join(base_path, 'feature/mirrored')

#print(output_dir)

for image_path in final_labels['path']:

  # read from the download path as kaggle likes to cache
  # so it isn't always where you tell it to go!

  source_full_path = os.path.join(download_path,image_path)

  print('source '+source_full_path)

  im = Image.open(source_full_path)

  # Get the side of the image (L or R) from final_labels
  # and flip the image only for Right side (to ensure all pointing one way)

  image_side = final_labels.loc[final_labels['path'] == image_path, 'side'].iloc[0]

  if (image_side == 'R'):
    im = im.transpose(method=Image.Transpose.FLIP_LEFT_RIGHT)

  # Construct the full path for the output file
  # and create intermediate or it fails

  #print(output_dir)

  output_path = os.path.join(output_dir, image_path)
  os.makedirs(output_path[:output_path.rindex(os.path.sep)], exist_ok=True)
  im.save(output_path)

  im.close()

### Image Trimming

Its quickly observable that the images contain different sizes. Some of this is due to white space, some because of breast size. Initially look to trim all of the black space out before normalising the images a little

In [ ]:
import matplotlib.pyplot as plt

# grab a quick freq diagram of the images to show the problem of size

image_sizes = pd.merge(image_details, final_labels, on='filename',how='left')

image_size_freq = image_sizes \
                      .groupby(['width','height'])['BIRAD'] \
                      .agg(['count'])


image_size_freq.plot.bar(figsize=(8,2))


# have a look at the aspect ratios so we can review trimming issues

image_sizes['aspect'] = round(image_sizes['height']/image_sizes['width'])

image_size_aspect = image_sizes \
                      .groupby(['aspect'])['aspect'] \
                      .agg(['count'])

# DEBUG display(trimmed_image_details)

image_size_aspect.plot.bar(figsize=(8,2))


this is particularly annoying as this long tail when you look at it by BI-RADS is actually almost all BIRAD 5 which is small enough.

The first issue is these BIRAD5 images are all trimmed to remove the "whitespace" on the image.

The other is largely that of size which. As a result, whilst its tempting to perhaps exclude the small images - this can't be done. Equally resizing the other images would squeeze the image to kill all future work. The best way therefore is to initially trim the images to remove the space and then to resize to a similar aspect so as not to lose information.


In [ ]:
## this gives a couple of examples of this
## yes I have resized, this is just to show the padding which exists
## on the BIRAD5 images

from PIL import Image, ImageShow

(width, height) = (im.width // 4, im.height // 4)

print(test_paths_b5[0])

image_path = os.path.join(base_path, 'feature/mirrored')

im = Image.open(image_path+'/'+test_paths_b1[0])
(width, height) = (im.width // 4, im.height // 4)
display(im.resize((width, height)))

im = Image.open(image_path+'/'+test_paths_b5[0])
(width, height) = (im.width // 4, im.height // 4)
display(im.resize((width, height)))

So what we are going to do is pre-process all the other images to trim all blacks out to the left. Now we have flipped all the Right images this should be a little easier

In [ ]:
# HAVE LEFT THIS IN HERE TO SHOW I DID THE NAIVE IMPLEMENTATION INITIALLY
# THIS ACTUALLY WORKED SOMEHOW FIRST TIME, BUT WAS REAAAAAALLLY SLOW AFTER
# GDRIVE I THINK THROTTLES YOU SO NEXT CELL IS THE SLICK IMPLEMENTATION
# JUST PAINFUL TO READ

#from PIL import Image
#import matplotlib.pyplot as plt

#input_dir = os.path.join(base_path, 'feature/mirrored')
#output_dir = os.path.join(base_path, 'feature/trimmed')


#for image_path in ['BIRAD1/b1/2017_BC009804_ MLO_R.jpg']: #final_labels['path']:

  #source_full_path = os.path.join(input_dir,image_path)

  #im = Image.open(source_full_path)

  #print('processing:'+source_full_path)

  # its an RBG, but this is much simpler given its greyscale anyway
  #im_grey = im.convert('L')

  # grab this as a pixel array so we can just iterate through it
  # and find the widths where we are finding all blacks and trim
  # two points of note where this has been adjusted:

  # - the value of "black" has been adjusted to "near black" for quality
  #   this is to avoid noise.

  # - we aren't looking at the overall width. After performing analysis
  #   after the first time on sizes, there are some images where the scan
  #   has small white band on the top right where it looks like the image
  #   was scanned with a very slight rotation, (2013_BC004406_ MLO_L) so
  #   ignoring the last 10px which solves, and doesn't damage other images

  # NOTE: getpixel is VERY slow - there will be a better way, just
  #       just can't find it right now. Have a feeling something is
  #       is being throttled as it generated quickly enough initially

  #blacks_column_count = []

  #for x in range(0,im_grey.width-20):
  #  black_count = 0

  #  for y in range(0,im_grey.height-1):
  #
  #    #print(im_grey.getpixel((x,y)))

  #   if (im_grey.getpixel((x,y)) > 30):
  #      black_count = black_count + 1
  #      #print(im_grey.getpixel((x,y)))

  #  if (black_count > 0):
  #    blacks_column_count.append(x)

  # DEBUG:
  # print(blacks_column_count)

  # do the cropping of the image based on where we don't blacks in a
  # vertical. This will trim both sides which we see in images

  #crop_left = min(blacks_column_count)
  #crop_right = max(blacks_column_count)
  #im_cropped = im.crop((crop_left,0,crop_right,im_grey.height))

  #output_path = os.path.join(output_dir, image_path)
  #os.makedirs(output_path[:output_path.rindex(os.path.sep)], exist_ok=True)
  #im_cropped.save(output_path)

  #im_cropped.close()
  #im.close()

In [ ]:
# get the images
# work out the height, and sum all black pixels across every vertical
# once we have that resize to remove all of this just leaving the breast only

from PIL import Image
import numpy as np

input_dir = os.path.join(base_path, 'feature/mirrored')
output_dir = os.path.join(base_path, 'feature/trimmed')


for image_path in final_labels['path']:

  source_full_path = os.path.join(input_dir,image_path)

  im = Image.open(source_full_path)

  # print('processing:'+source_full_path)

  # grab this as a pixel array so we can just iterate through it
  # and find the widths where we are finding all blacks and trim
  # two points of note where this has been adjusted:

  # - the value of "black" has been adjusted to "near black" for quality
  #   this is to avoid noise.

  # - we aren't looking at the overall width. After performing analysis
  #   after the first time on sizes, there are some images where the scan
  #   has small white band on the top right where it looks like the image
  #   was scanned with a very slight rotation, (2013_BC004406_ MLO_L) so
  #   ignoring the last 10px which solves, and doesn't damage other images

  # NOTE: getpixel is VERY slow - there will be a better way, just
  #       just can't find it right now. Have a feeling something is
  #       is being throttled as it generated quickly enough initially

  # UPDATE: this was way more painful that probably ought to

  im_bw = im.convert('L').getchannel('L')

  img_height = im_bw.size[1]
  img_width = im_bw.size[0]

  #print('width:' +str(height))
  #print('height:' +str(width))

  # get data gets all the data into a flat list
  # this gets then reshaped into the image height and width as arrays
  im_data = np.array(im_bw.getdata())
  im_array = im_data.reshape(img_height, img_width)

  # we aren't going to take 0 as black, but 30 is black enough
  # it avoids a load of noise in the images
  # set it to be binary just to help
  im_array_black = np.where(im_array < 31, 0, 1)

  # now sum across the columns in the array
  # this will give 0 where all black
  im_array_trim = im_array_black.sum(axis=0)

  # again cute numpy hacks to find the min and max columns
  # where there aren't 0 sums, ie. they are part of the image
  # where the [::-1] is reversing and counting back for the max
  crop_left = (im_array_trim!=0).argmax(axis=0)
  crop_right = img_width-((im_array_trim[::-1])[20:]!=0).argmax(axis=0)

  # now just do the crop and save it out
  im_cropped = im.crop((crop_left,0,crop_right,im_bw.height))

  output_path = os.path.join(output_dir, image_path)
  os.makedirs(output_path[:output_path.rindex(os.path.sep)], exist_ok=True)
  im_cropped.save(output_path)

  im_cropped.close()
  im.close()



In [ ]:
# test again after trimming

from PIL import Image
import matplotlib.pyplot as plt

trimmed_image_details = pd.DataFrame(columns=('path','type'))

for image_path in final_labels['path']:

  source_full_path = os.path.join(base_path,'feature/trimmed',image_path)

  im = Image.open(source_full_path)

  im_details = pd.DataFrame({
                                'file':[im.filename],
                                'width':[im.width],
                                'height':[im.height],
                            })
  trimmed_image_details = pd.concat([trimmed_image_details, im_details])
  im.close()


# grab a quick freq diagram of the images to show the problem of size

trimmed_image_details['aspect'] = round(trimmed_image_details['height']/trimmed_image_details['width'])

trimmed_image_size_freq = trimmed_image_details \
                      .groupby(['aspect'])['aspect'] \
                      .agg(['count'])

# DEBUG
#display(trimmed_image_size_freq)

trimmed_image_size_freq.plot.bar(figsize=(8,2))

## Normalise the sizes

Whilst its possible to add CNN filters of abnormal sizes now that that images are more common, we can go ahead and just resize these.

In [ ]:
# grab the images and resize to standard aspect ratio.
# following the trimming this is now around 2 on average
# as we saw which should allow for full images without sever distortion



from PIL import Image
import matplotlib.pyplot as plt

input_dir = os.path.join(base_path, 'feature/sized')
output_dir = os.path.join(base_path, 'feature/sized')

# switching to a standard size .. this makes the CNN way easier
# the distortion is very minimal by keeping the aspect ratio

normal_height = 600 ## 578 is the standard height
normal_width = 300 ## 280 is width to hit a 2:1 aspec ratio


for image_path in final_labels['path']:

  source_full_path = os.path.join(input_dir,image_path)

  im = Image.open(source_full_path)

  # resize to the chosen size. BICUBIC - high quality resample
  im_resized = im.resize((normal_width, normal_height), resample=Image.BICUBIC)

  output_path = os.path.join(output_dir, image_path)
  os.makedirs(output_path[:output_path.rindex(os.path.sep)], exist_ok=True)
  im_resized.save(output_path)

  im_resized.close()
  im.close()


# Feature Engineering

So we know that there are a few different areas to look at here.


1. The category of the breast tissue

If the tissue is dense rather than fatty then the breast images will show up with more "white". Obviously at a simple level we are looking for white masses in a darker image then this is more problematic and needs to be factored in

2. Potentially we may need to adjust the images. The actual breast image will be a

3. We have for every patient 4 images - Left and Right breast and two different scans - MLO and CC. We would expect a correlation of results here. There ought to be a strong correlation between the scans on the same breast and a smaller correlation between breasts (since there are genetic factors if you have it in one breast there is a higher chance in the other)

4. We may also want to aid recognition altering the orientation of the images so that left points the same way as right.

## Feature: Breast Density

When we are looking at mammograms one of the other large features we need to consider is the type of tissue. Fat will show as darker in the images, and tissue is bright. This is particularly important when looking for a tumor which as a set of dense tissue cells will appear more apparent against a dark background.

Whilst some datasets have this already this dataset does not. In order to provide an input we will generate a feature to try and estimate this. It will be a feature per image. The way we will do this is by reviewing the histogram of the images.

In [ ]:
# we can grab out the image type, size and format to verify
# Then we can start to assess the % of remaining pixels. This isn't completely ideal. There are some images where the chest
# cavity is partially visible (and a couple where obviously an implant is visible), but should give a better set of data to assess this

from PIL import Image
import matplotlib.pyplot as plt
import numpy as np


density_data = []
breast_density = None

for image_path in final_labels['path']:

  source_full_path = os.path.join(base_path,'feature/sized',image_path)

  im = Image.open(source_full_path)

  # the images are actually RBG so converting this to greyscale to make this more sensible

  im_grey = im.convert('L')
  grey_freq = im_grey.getcolors()
  im.close()
  df_greyfreq = pd.DataFrame(grey_freq, columns=['frequency','colour'])
  #print(df_greyfreq)

  # most of the image is actually black, so really we want to exclude this - or general noise of the image
  # so filter out black or near - [0-10] as greyscale which is being used for the trimming.

  pixels_notblack = df_greyfreq.loc[df_greyfreq['colour']>30].copy()

  # create a binned frequency table with % across the bins to normalise across different
  # image sizes and also breast sizes. We are cutting into

  pixels_notblack["greyscale_bin"] = pixels_notblack["colour"] // 25
  df_sumbins = pixels_notblack.groupby(['greyscale_bin'],observed=False).sum()

  # in order to normalise over the total pixels. We have different sizes of every factor
  # image, breast in image (we have not excluded blacks), breast this should normalise this feature
  # obviously we have removed < 30 - so it will be 1-10

  total_pixels_grey = pixels_notblack['frequency'].sum()
  df_sumbins_prc = round(100*(df_sumbins['frequency']/total_pixels_grey),2)
  df_sumbins_prc['image_path'] = image_path

  # put this data now into a numpy array transposed so we get a nice
  # array of bucketed greyscale values along with the label

  row_frame = pd.DataFrame(df_sumbins_prc).transpose()

  breast_density = pd.concat([breast_density,row_frame],axis=0)


# don't need this label here
feature_breast_density = breast_density




In [ ]:

# we have now the feature we will use
display(feature_breast_density.columns)


Index([1, 2, 3, 4, 5, 6, 7, 8, 9, 'image_path', 10], dtype='object', name='greyscale_bin')

## Feature: Scan and Patient

This includes the patient id as well as the image type and

In [ ]:
# we only need a few of the total labels

# path - to join up the data
# patient, imaging_type, side all provide a

feature_labels = final_labels[['path','patient','imaging_type','side']].copy()

#print(len(feature_labels))
feature_labels.head()


# this is all categorical data, so doing OHE on it, but

final_feature_density = pd.get_dummies(feature_labels,columns=['patient','imaging_type','side'])

final_feature_density.head()

# check same number of rows, so we are good (more TDD than anything)!
# print(len(one_hot_density))

assert(len(feature_labels) == len(final_feature_density))


# Final analysis before Execution


## Distribution of BIRADs

In [ ]:
# quick check of the images here. There aren't a *lot* of images which are positive in fairness.
# This is a bit of a risk in the efficacy if we are looking for features. Probably a better approach
# then is to look at algorithms which are trained to look for outliers and use the BIRAD 1 as the
# training dataset initially. Given 3,4 are suspicious we need to look how to treat these

final_labels.groupby('BIRAD')['BIRAD'].count()

,BIRAD
BIRAD,
1,1839
3,387
4,100
5,20


# Model Creation

Because of the distribution of the training data we are going to look for outliers in the base data. In this case its easier to train then the data on "normal" and run against exceptions, rather than a classifier model where we require large amount of data. We aren't so much looking to partition the space or look for features, but to search for particular abnormalies.

This also fits with the procedure where medically only a biopsy is a confirmation - everything is effectively just searching for something unusual to review further.

As a result unsupervised autoencoder model approach would intuitively seem to be the best approach here.

## Feature dataset

## Training partitioning

Start by doing the normal partitioning of the training set

In [ ]:
from torchvision import transforms, datasets
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToTensor

# we are trying to detect abnormalities, so training only on BIRAD1 as our large "normal" set
# stripping off b1 as it expects the classification in the folder structure when loading in.

source_full_path = os.path.join(base_path,'feature/sized/BIRAD1')

# note that in alternative tests I've split the BIRAD1 into test and train as well
# to validate - however in this final its left together to compare against BIRAD3-5

# we've pre-processed everything carefully, so just a case of changing to tensor on load
toTensor=transforms.ToTensor()

train_dataset = datasets.ImageFolder(root=source_full_path, transform=toTensor)
train_loader = DataLoader(dataset=train_dataset, batch_size=100, shuffle=True)

## Model Definition

In [ ]:
import torch.nn as nn
import torchvision
import torch.nn.functional as F

class AENet(nn.Module):
    def __init__(self):
        super().__init__()

        ## convert to a 2d matrix.

        ## We have reduced to greyscale so 1 channel and keeps computation lower
        ## Created images which are 300x600

        ## The step down for the autoencoder model has been left
        ## as fairly generic based on various. Additional changes have been
        ## tried with different options, however this actually converges
        ## and is on the limit of the CoLab TPU memory with a small epoch range

        ## the result is a pretty simple step down of feature with a few
        ## hidden layers and then decoding and a final activation function.

        ## ReLu is in here as fairly standard - although note that it is very
        ## fast

        image_height = 600
        image_width = 300

        self.encoder = nn.Sequential(
            nn.Linear(600*300, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 36),
            nn.ReLU(),
            nn.Linear(36, 18),
            nn.ReLU(),
            nn.Linear(18, 9)
        )
        self.decoder = nn.Sequential(
            nn.Linear(9, 18),
            nn.ReLU(),
            nn.Linear(18, 36),
            nn.ReLU(),
            nn.Linear(36, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 600*300),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded


In [ ]:
### IMPORTANT - THIS IS AN ALTERNATIVE MODEL TESTED

## This model cuts down the size of the NN and reduces the layers to test the
## impact of small NN due to resource constraints and whether a larger
## number of training iterations can be used as a result as well

import torch.nn as nn
import torchvision
import torch.nn.functional as F

class AENetSmall(nn.Module):
    def __init__(self):
        super().__init__()

        ## convert to a 2d matrix.

        ## this is an alternative

        image_height = 600
        image_width = 300

        self.encoder = nn.Sequential(
            nn.Linear(600*300, 64),
            nn.ReLU(),
            nn.Linear(64, 16),
            nn.ReLU(),
            nn.Linear(16, 8)
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 64),
            nn.ReLU(),
            nn.Linear(64, 600*300),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded


## Model Training

In [ ]:
#iterate through rounds to train the model
import torch
import torch.optim as optim

mammogram_cnn = AENet()

# number of epochs has been basically limited to 5
# due to constraints on the colab runtime.
# It could be optimised potentially -
number_of_epochs = 5
outputs = []
losses = []

# MSELoss is being used here which is pretty standard

# Adam is also being used as an optimiser. Given alternatives this
# works better with smaller datasets and with dynamic gradient is supposed
# to avoid getting trapped in local minima on decent

loss_function = nn.MSELoss()
optimizer = optim.Adam(mammogram_cnn.parameters(), lr=1e-3, weight_decay=1e-8)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mammogram_cnn.to(device)

for epoch in range(number_of_epochs):  # loop over the dataset multiple times

  for images, _ in train_loader:
      images = images.view(-1, 600*300).to(device)

      reconstructed = mammogram_cnn(images)
      loss = loss_function(reconstructed, images)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      losses.append(loss.item())

      outputs.append((epoch, images, reconstructed))

  print(f"Epoch {epoch+1}/{number_of_epochs}, Loss: {loss.item():.6f}")

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt



plt.style.use('fivethirtyeight')
plt.figure(figsize=(8, 5))
plt.plot(losses, label='Loss')
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.legend()
plt.show()

### Save the model

In [ ]:
# save the model to the output path

model_path = os.path.join(base_path, 'model/mammogram_aenn.torch')

os.makedirs(model_path[:model_path.rindex(os.path.sep)], exist_ok=True)

torch.save(mammogram_cnn.state_dict(), model_path)

### Hyper parameters

batch size of 100 seems to be optimal. Anything bigger and its not converging quickly anything larger doesn't converge enough

I'm rather limited with the epochs because of the memory and limits on resources in colo - and whilst I'm rather using default gradients based on some of the examples doing further Hyper Parameter optimisation is unrealistic unfortunately.

Having said that it does seem to converge nicely with these params

# Analysis

## Load the model

This is mainly to save time and avoid running the entire notebook (other than running model definition).

Once features are concatenated in then features are also required as well

In [ ]:
# go and load the model - it just saves time when changing things and we aren't going
# to redo the data or train every time we want to test/do inference
# again some of this is so that I can run parts of the notebook only so repeated imports etc.

import torch
import torch.nn as nn
import torchvision
import os

model_path = os.path.join(base_path, 'model/mammogram_aenn.torch')

# load the model and the state we saved

mammogram_model = AENet()
mammogram_model.load_state_dict(torch.load(model_path))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mammogram_model.to(device)


# check its loaded ok
mammogram_model.eval()


AENet(
  (encoder): Sequential(
    (0): Linear(in_features=180000, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=36, bias=True)
    (5): ReLU()
    (6): Linear(in_features=36, out_features=18, bias=True)
    (7): ReLU()
    (8): Linear(in_features=18, out_features=9, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=9, out_features=18, bias=True)
    (1): ReLU()
    (2): Linear(in_features=18, out_features=36, bias=True)
    (3): ReLU()
    (4): Linear(in_features=36, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=128, bias=True)
    (7): ReLU()
    (8): Linear(in_features=128, out_features=180000, bias=True)
    (9): Sigmoid()
  )
)

## Model Testing

In [ ]:
from torchvision import transforms, datasets
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToTensor

# now we want to load other Birads which are the confirmed and see what happens!
# we've pre-processed everything carefully, so just a case of changing to tensor on load

toTensor=transforms.ToTensor()

birads_1_path = os.path.join(base_path,'feature/sized/BIRAD1')
birads_3_path = os.path.join(base_path,'feature/sized/Birad3')
birads_4_path = os.path.join(base_path,'feature/sized/Birad4')
birads_5_path = os.path.join(base_path,'feature/sized/Birad5')


test_dataset = torchvision.datasets.ImageFolder(root=birads_1_path, transform=toTensor)
test_birad_1_loader = DataLoader(dataset=test_dataset, batch_size=10, shuffle=True)

test_dataset = torchvision.datasets.ImageFolder(root=birads_3_path, transform=toTensor)
test_birad_3_loader = DataLoader(dataset=test_dataset, batch_size=10, shuffle=True)

test_dataset = torchvision.datasets.ImageFolder(root=birads_4_path, transform=toTensor)
test_birad_4_loader = DataLoader(dataset=test_dataset, batch_size=10, shuffle=True)

test_dataset = torchvision.datasets.ImageFolder(root=birads_5_path, transform=toTensor)
test_birad_5_loader = DataLoader(dataset=test_dataset, batch_size=10, shuffle=True)


torch.inference_mode()

In [ ]:
## run the predictions against all BIRAD values
## this really isn't the prettiest piece of code, but

import torch.nn.functional as functional

test_loss = 0
correct = 0

predictions_list_1 = []
predictions_list_3 = []
predictions_list_4 = []
predictions_list_5 = []

print("processing predictions for BIRAD1")

for images, _ in test_birad_1_loader:

    images = images.view(-1, 600*300).to(device)

    #actually execute the prediction
    prediction = mammogram_model(images)

    # grab the mse loss and capture than in an array so we can do some analysis after
    prediction_out = (functional.mse_loss(prediction,images))
    #print(prediction_out.detach().numpy())


    # adding the cpu to try and make sure this all works between [T,G,C]PU
    # as colob is being a real pain on resources

    loss = prediction_out.cpu().detach().numpy().tolist()

    predictions_list_1.append(loss)

print("processing predictions for BIRAD3")

for images, _ in test_birad_3_loader:

    images = images.view(-1, 600*300).to(device)

    #actually execute the prediction
    prediction = mammogram_model(images)

    # grab the mse loss and capture than in an array so we can do some analysis after
    prediction_out = (functional.mse_loss(prediction,images))
    #print(prediction_out.detach().numpy())


    # adding the cpu to try and make sure this all works between [T,G,C]PU
    # as colob is being a real pain on resources

    loss = prediction_out.cpu().detach().numpy().tolist()

    predictions_list_3.append(loss)

for images, _ in test_birad_4_loader:

    print("processing predictions for BIRAD4")

    images = images.view(-1, 600*300).to(device)

    #actually execute the prediction
    prediction = mammogram_model(images)

    # grab the mse loss and capture than in an array so we can do some analysis after
    prediction_out = (functional.mse_loss(prediction,images))
    #print(prediction_out.detach().numpy())


    # adding the cpu to try and make sure this all works between [T,G,C]PU
    # as colob is being a real pain on resources

    loss = prediction_out.cpu().detach().numpy().tolist()

    predictions_list_4.append(loss)


print("processing predictions for BIRAD5")

for images, _ in test_birad_5_loader:


    images = images.view(-1, 600*300).to(device)

    #actually execute the prediction
    prediction = mammogram_model(images)

    # grab the mse loss and capture than in an array so we can do some analysis after
    prediction_out = (functional.mse_loss(prediction,images))

    # adding the cpu to try and make sure this all works between [T,G,C]PU
    # as colob is being a real pain on resources

    loss = prediction_out.cpu().detach().numpy().tolist()

    predictions_list_5.append(loss)


processing predictions for BIRAD1
processing predictions for BIRAD3
processing predictions for BIRAD4
processing predictions for BIRAD4
processing predictions for BIRAD4
processing predictions for BIRAD4
processing predictions for BIRAD4
processing predictions for BIRAD4
processing predictions for BIRAD4
processing predictions for BIRAD4
processing predictions for BIRAD4
processing predictions for BIRAD4
processing predictions for BIRAD5


In [ ]:
## We should now have dataframes with all of the predictions in
## including running BIRAD1 as the baseline as it was used for training

#pd.DataFrame(predictions_list_1).describe()
pd.DataFrame(predictions_list_3).describe()
#pd.DataFrame(predictions_list_4).describe()
#pd.DataFrame(predictions_list_5).describe()

,0
count,1.000000
mean,0.020454
std,NaN
min,0.020454
25%,0.020454
50%,0.020454
75%,0.020454
max,0.020454
